## Scraping

In [1]:
import requests
import base64
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

BASE_URL = "https://www.singaporepools.com.sg/en/product/pages/4d_results.aspx?sppl="

session = requests.Session()

rows = []

START_DRAW = 5452
STOP_DRAW = 4983

draw = START_DRAW
fail_count = 0

while draw >= STOP_DRAW:

    encoded = base64.b64encode(f"DrawNumber={draw}".encode()).decode()
    url = BASE_URL + encoded

    print(f"Scraping draw {draw}")

    try:
        r = session.get(url, timeout=10)
    except requests.RequestException:
        fail_count += 1
        if fail_count > 20:
            break
        draw -= 1
        continue

    if r.status_code != 200:
        fail_count += 1
        if fail_count > 20:
            break
        draw -= 1
        continue

    soup = BeautifulSoup(r.text, "html.parser")

    # -------------------------
    # Extract draw date
    # -------------------------

    draw_date = None
    page_text = soup.get_text(" ", strip=True)

    date_match = re.search(
        r"\b(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun),\s\d{2}\s\w{3}\s\d{4}",
        page_text
    )

    if date_match:
        try:
            draw_date = pd.to_datetime(date_match.group(0))
        except:
            draw_date = None

    # -------------------------
    # Extract 4D numbers
    # -------------------------

    numbers = []

    for td in soup.find_all("td"):

        txt = td.get_text(strip=True)

        if txt.isdigit() and len(txt) == 4:
            numbers.append(txt)

    # Ensure correct count
    if len(numbers) < 23:

        fail_count += 1

        if fail_count > 20:
            print("Too many failures, stopping")
            break

        draw -= 1
        continue

    fail_count = 0

    row = {
        "draw_number": draw,
        "draw_date": draw_date,

        "first_prize": numbers[0],
        "second_prize": numbers[1],
        "third_prize": numbers[2],

        "starter_1": numbers[3],
        "starter_2": numbers[4],
        "starter_3": numbers[5],
        "starter_4": numbers[6],
        "starter_5": numbers[7],
        "starter_6": numbers[8],
        "starter_7": numbers[9],
        "starter_8": numbers[10],
        "starter_9": numbers[11],
        "starter_10": numbers[12],

        "consolation_1": numbers[13],
        "consolation_2": numbers[14],
        "consolation_3": numbers[15],
        "consolation_4": numbers[16],
        "consolation_5": numbers[17],
        "consolation_6": numbers[18],
        "consolation_7": numbers[19],
        "consolation_8": numbers[20],
        "consolation_9": numbers[21],
        "consolation_10": numbers[22],
    }

    rows.append(row)

    draw -= 1

    time.sleep(0.2)

# -------------------------
# Convert to dataframe
# -------------------------

df = pd.DataFrame(rows)

# Convert draw date
df["draw_date"] = pd.to_datetime(df["draw_date"])

# Remove duplicates
df = df.drop_duplicates(subset="draw_number")

# Sort
df = df.sort_values("draw_number")

# -------------------------
# Save CSV
# -------------------------

df.to_csv("singapore_4d_history.csv", index=False)

print("Scraping complete")
print("Total draws saved:", len(df))

Scraping draw 5452
Scraping draw 5451
Scraping draw 5450
Scraping draw 5449
Scraping draw 5448
Scraping draw 5447
Scraping draw 5446
Scraping draw 5445
Scraping draw 5444
Scraping draw 5443
Scraping draw 5442
Scraping draw 5441
Scraping draw 5440
Scraping draw 5439
Scraping draw 5438
Scraping draw 5437
Scraping draw 5436
Scraping draw 5435
Scraping draw 5434
Scraping draw 5433
Scraping draw 5432
Scraping draw 5431
Scraping draw 5430
Scraping draw 5429
Scraping draw 5428
Scraping draw 5427
Scraping draw 5426
Scraping draw 5425
Scraping draw 5424
Scraping draw 5423
Scraping draw 5422
Scraping draw 5421
Scraping draw 5420
Scraping draw 5419
Scraping draw 5418
Scraping draw 5417
Scraping draw 5416
Scraping draw 5415
Scraping draw 5414
Scraping draw 5413
Scraping draw 5412
Scraping draw 5411
Scraping draw 5410
Scraping draw 5409
Scraping draw 5408
Scraping draw 5407
Scraping draw 5406
Scraping draw 5405
Scraping draw 5404
Scraping draw 5403
Scraping draw 5402
Scraping draw 5401
Scraping dra

## Machine Learning 

In [2]:
import pandas as pd # Load data manipulation library
import matplotlib.pyplot as plt # Load data visualization library
import seaborn as sns # Load statistical data visualization library
import numpy as np # Load numerical computation library

# Initialize machine learning models
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score , root_mean_squared_error
import joblib # Used to save models

In [2]:
df = pd.read_csv('singapore_4d_history.csv')
df

,draw_number,draw_date,first_prize,second_prize,third_prize,starter_1,starter_2,starter_3,starter_4,starter_5,...,consolation_1,consolation_2,consolation_3,consolation_4,consolation_5,consolation_6,consolation_7,consolation_8,consolation_9,consolation_10
0,4983,5/3/2023,8013,8140,2276,72,332,903,1623,2893,...,455,1441,1809,2650,3486,5487,6404,7751,9512,9930
1,4984,8/3/2023,7493,8400,1971,1318,2126,3438,4030,4921,...,1671,2339,2530,2829,3666,4690,4737,5187,8288,8732
2,4985,11/3/2023,7982,4760,212,342,613,1404,4645,5956,...,1073,3586,3648,4233,5933,6269,7026,7154,8558,9256
3,4986,12/3/2023,691,2922,1822,1221,2099,3878,4006,4162,...,972,1279,3310,3321,3424,4343,5862,5952,9139,9785
4,4987,15/3/2023,1905,8878,5744,741,1428,2654,3198,3573,...,250,277,677,804,1030,1535,2762,3114,4705,8981
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465,5448,22/2/2026,1903,7215,3568,41,1203,1695,3950,6171,...,1376,1790,2634,3301,4698,5217,5685,7875,9419,9984
466,5449,25/2/2026,1516,7309,2708,814,1068,3542,3847,4163,...,1511,2357,3115,4586,4965,6034,6445,6473,7266,8196
467,5450,28/2/2026,4172,198,5482,1002,3679,4256,4411,4486,...,433,515,2090,2546,4101,4473,4558,6202,8441,9542
468,5451,1/3/2026,3927,7192,2607,318,871,1083,1698,1854,...,483,1995,2792,4408,6217,7338,7340,7953,9154,9595


In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [6]:
import os
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

# -----------------------
# CONFIG
# -----------------------

CSV_FILE = "singapore_4d_history.csv"
RESULTS_URL = "https://www.singaporepools.com.sg/en/product/Pages/4d_results.aspx"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

np.random.seed(42)

# -----------------------
# HELPER FUNCTIONS
# -----------------------

def normalize_probs(p):
    """Normalize an array to sum to 1 and clip negative values."""
    p = np.array(p)
    p = np.maximum(p, 0)
    s = p.sum()
    return p / s if s > 0 else np.ones(len(p)) / len(p)

def get_latest_website_date():
    """Scrape the latest draw date from Singapore Pools website."""
    try:
        r = requests.get(RESULTS_URL, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        date_element = soup.find(
            "span",
            {"id":"ctl00_ctl37_g_1c8ad7f0_9b2f_4f9a_ba9f_5f6e3b3c2c40_lblDrawDate"}
        )
        if date_element:
            return pd.to_datetime(date_element.text.strip(), errors="coerce")
    except Exception as e:
        print("Website check failed:", e)
    return None

# -----------------------
# CHECK IF DATA UPDATED
# -----------------------

df = pd.read_csv(CSV_FILE)
latest_csv_date = pd.to_datetime(df["draw_date"]).max()
latest_web_date = get_latest_website_date()

if latest_web_date is not None and latest_csv_date >= latest_web_date:
    print("Data already up to date. Exiting...")
    exit()

print("New draw detected. Running prediction pipeline...\n")

# -----------------------
# PREP DATA
# -----------------------

number_cols = [c for c in df.columns if c not in ["draw_number","draw_date"]]
df["numbers"] = df[number_cols].apply(lambda row: [str(n).zfill(4) for n in row], axis=1)

all_numbers = []
for _, row in df.iterrows():
    all_numbers.extend(row["numbers"])

dataset = pd.DataFrame({"number": all_numbers})
dataset["d1"] = dataset["number"].str[0].astype(int)
dataset["d2"] = dataset["number"].str[1].astype(int)
dataset["d3"] = dataset["number"].str[2].astype(int)
dataset["d4"] = dataset["number"].str[3].astype(int)

# -----------------------
# PROBABILISTIC FEATURES
# -----------------------

windows = [20, 50, 100, 200]
digit_probs = {}

for pos in ["d1","d2","d3","d4"]:
    probs = np.zeros(10)
    for w in windows:
        hist = dataset[pos].iloc[-w:]
        freq = hist.value_counts(normalize=True).reindex(range(10), fill_value=0)
        probs += freq.values
    digit_probs[pos] = normalize_probs(probs / len(windows))

# Add recency weighting (boost numbers that haven't appeared recently)
for pos in ["d1","d2","d3","d4"]:
    last_idx = {d: (dataset[dataset[pos]==d].index.max() if d in dataset[pos].values else -999) for d in range(10)}
    recency_weight = np.array([1 / (1 + len(dataset) - last_idx[d]) for d in range(10)])
    digit_probs[pos] = normalize_probs(0.8 * digit_probs[pos] + 0.2 * recency_weight)

# -----------------------
# MONTE CARLO SAMPLING
# -----------------------

predictions = []
n_samples = 50000
for _ in range(n_samples):
    d1 = np.random.choice(np.arange(10), p=digit_probs["d1"])
    d2 = np.random.choice(np.arange(10), p=digit_probs["d2"])
    d3 = np.random.choice(np.arange(10), p=digit_probs["d3"])
    d4 = np.random.choice(np.arange(10), p=digit_probs["d4"])
    predictions.append(f"{d1}{d2}{d3}{d4}")

ranked = pd.Series(predictions).value_counts()
result = ranked.head(50).reset_index()
result.columns = ["number","freq"]

print("\nTop 50 predicted numbers\n")
print(result)

result.to_csv(f"{MODEL_DIR}/top_predictions.csv", index=False)
print(f"\nPredictions saved to {MODEL_DIR}/top_predictions.csv")

New draw detected. Running prediction pipeline...


Top 50 predicted numbers

   number  freq
0    1251    46
1    1258    42
2    8208    41
3    7201    40
4    8251    38
5    8201    38
6    8108    36
7    8181    35
8    7258    34
9    1101    34
10   7151    33
11   7158    33
12   8698    33
13   1208    32
14   8281    32
15   8257    32
16   8291    32
17   7698    31
18   8081    31
19   3288    31
20   7298    31
21   8258    31
22   8151    31
23   8358    30
24   1298    30
25   7608    30
26   7291    29
27   1651    29
28   8191    29
29   8157    29
30   1218    29
31   8209    29
32   3151    29
33   7188    29
34   8188    29
35   8351    29
36   3209    29
37   8254    29
38   3101    28
39   1688    28
40   7281    27
41   8198    27
42   7208    27
43   8311    27
44   7152    27
45   7251    27
46   8289    27
47   8607    27
48   8581    27
49   5251    27

Predictions saved to models/top_predictions.csv
